## Ripple data analysis

Contains Ripple data collected from Gian over two sessions.


In [51]:
# Loading/Import related packages
import sys
import os

#Usual suspects
import pandas as pd
import numpy as np
import json
import pickle as pkl
import matplotlib.pyplot as plt

#Extras for plotting
from matplotlib.patches import Patch

#Extra for typing
from collections import defaultdict

# Needed to point the importer to the src folder
sys.path.insert(0,r"C:\Users\annas\SynologyDrive\MedUniWien\Projects\NeuroClasp\Students\Liz Kalenteridis\Ripple")


#Decomposition/Processing Imports
from src.muniverse.algorithms.cbss import CBSS

In [ ]:
REPO_DIR  = os.path.abspath(os.getcwd())
INPUT_DIR = os.path.join(REPO_DIR, 'data')
OUTPUT_DIR = os.path.join(REPO_DIR, 'results')

combined_data_1_path = os.path.join(INPUT_DIR, 'emg_recording_giuan_tewst_20260826_122524.pkl')
session1_folder = os.path.join(INPUT_DIR, 'emg_recording_giuan_tewst_20260826_122524_divided')
session2_folder = os.path.join(INPUT_DIR, 'emg_recording_giuan_tewst2_20260826_135844_divided')

result_pandas = pd.read_pickle(combined_data_1_path)
print(result_pandas.keys())
print(result_pandas['trial_metadata']['items'])
# print(result_pandas.items())



dict_keys(['data', 'srate', 'n_channels', 'filtered', 'pre_trigger_seconds', 'timestamp', 'stream_type', 'trial_metadata', 'session_timeline'])
[{'index': 0, 'model': 'fist.glb', 'animation': 0, 'repetitions': 8}, {'index': 1, 'model': 'fasttripodpinch.glb', 'animation': 0, 'repetitions': 8}, {'index': 2, 'model': 'fastfingerext.glb', 'animation': 0, 'repetitions': 8}]


In [63]:
print(repr(INPUT_DIR))
print(glob.glob(os.path.join(INPUT_DIR, '*')))
print(glob.glob(os.path.join(session1_folder, '*'))[:5])

'c:\\Users\\annas\\SynologyDrive\\MedUniWien\\Projects\\NeuroClasp\\Students\\Liz Kalenteridis\\Ripple\\data'
['c:\\Users\\annas\\SynologyDrive\\MedUniWien\\Projects\\NeuroClasp\\Students\\Liz Kalenteridis\\Ripple\\data\\emg_recording_giuan_tewst2_20260826_135844.pkl', 'c:\\Users\\annas\\SynologyDrive\\MedUniWien\\Projects\\NeuroClasp\\Students\\Liz Kalenteridis\\Ripple\\data\\emg_recording_giuan_tewst2_20260826_135844_divided', 'c:\\Users\\annas\\SynologyDrive\\MedUniWien\\Projects\\NeuroClasp\\Students\\Liz Kalenteridis\\Ripple\\data\\emg_recording_giuan_tewst_20260826_122524.pkl', 'c:\\Users\\annas\\SynologyDrive\\MedUniWien\\Projects\\NeuroClasp\\Students\\Liz Kalenteridis\\Ripple\\data\\emg_recording_giuan_tewst_20260826_122524_divided']
['c:\\Users\\annas\\SynologyDrive\\MedUniWien\\Projects\\NeuroClasp\\Students\\Liz Kalenteridis\\Ripple\\data\\emg_recording_giuan_tewst_20260826_122524_divided\\idle_001.pkl', 'c:\\Users\\annas\\SynologyDrive\\MedUniWien\\Projects\\NeuroClasp\\St

#### Task Parameters

In [84]:
# This task used 3 tasks, each with 8 repitions
n_reps = 8
n_tasks = 3
movement_names = ['fist', 'fast_tripod', 'fast_finger_ext']

samp_freq = 2000

In [55]:
# def load_simulation_data(input_dir, filename):

#     """
#     Load the simulation data from a pickle file.

#     Parameters:
#     - input_dir: str, the directory where the pickle file is located.
#     - filename: str, the name of the pickle file.

#     Returns:
#     - emg_array: dict, the loaded simulation EMG data. 
#     Normally contains 320 channels, here downsampled to the first 8x8 array to make decomposition faster .
#     """
#     file_path = os.path.join(input_dir, filename)

#     # Check if the file exists before attempting to load it
#     if not os.path.exists(file_path):
#         raise FileNotFoundError(f"The file {file_path} does not exist. Please ensure the input directory and filename are correct.")

#     result_pandas = pd.read_pickle(file_path)
#     print("Loaded data from:", file_path)
#     print(result_pandas.keys())
#     print(result_pandas['data'].shape)

#     emg_array = result_pandas['data']
#     #These two arrays contain the intended poses and durations for each pose
#     # We will use this information later in the analysis, see below in the Decomposition Section
#     poses = result_pandas['metadata']['poses_intended']
#     durations = result_pandas['metadata']['durations']


#     #NOTE: a tuple, not a set: a set has no order, so the unpacking below would be random
#     return emg_array, poses, durations
    

In [ ]:
import glob 

def get_task_data(session_dir, n_reps, movement_names, file_type):
    """
    Concatenate EMG data from multiple pickle files.

    Parameters:
    - pckl_files: list of str, the names of the pickle files to concatenate.
    - input_dir: str, the directory where the pickle files are located.
    - movement_names: list of str, names of movements sorted in the order of collection in task
    - file_type: str, the type of files to load (e.g., 'move' for movement files, or 'rest' for rest files).


    Returns:
    - movement_dict: dictionary with keys for each movement type
        Each key contains dataframe for each repetition with their respective emg data and durations 
    """
    #move_files =  sorted(glob.glob(os.path.join(input_dir + '*/move*')))
    move_files =  sorted(glob.glob(os.path.join(session_dir, f'*{file_type}*')))


    movement_dict= {}
    for movement_num, movement_type in enumerate(movement_names):
        start = movement_num * n_reps
        end = start + n_reps
        task_files = move_files[start:end]

        task_data = [pd.read_pickle(file_name) for file_name in task_files] 

        movement_dict[movement_type] = {
            'emg_data_list': [pd.DataFrame(r['data']) for r in task_data], 
            'movement_dur': [r['duration_s'] for r in task_data] 
        }

    return movement_dict

def get_move(movement_data, sampling_freq, win = None):
    """

    Paramters:
    - movement_data: dictionary containing keys corresponding to movement types.
    - sampling_freq: float, sampling frequency of data collection in Hz
    - win: float, optional, window size in seconds to extract from the middle of the movement duration. If None, entire rep duration is used.

    Returns:
    - iso_dict: dict containing each movement as keys, containing:
        - emg_iso_reps: list of dataframes, each containing the EMG data for a single repetition of the movement
        - emg_iso_combined: dataframe, containing the concatenated (wide) EMG data for all repetitions of the movement
        - movement_dur: list of floats, each representing the duration of a single repetition, passed through
    """
    half_win = win/2 if win is not None else None
    iso_dict = {}

    for movement, data in movement_data.items():
        emg_reps = []

        for emg_data, dur in zip(data['emg_data_list'], data['movement_dur']):
            if win is None:
                rep_data = emg_data # no windowing, take entire rep duration
            else:
                iso_start = int(((dur/2)-half_win) * sampling_freq)
                iso_end = int(((dur/2)+half_win) * sampling_freq)
                rep_data = emg_data.iloc[:, iso_start:iso_end]

            emg_reps.append(rep_data)

        iso_dict[movement]= {
            'emg_reps': emg_reps,
            'combined_emg_reps': pd.concat(emg_reps, ignore_index = True, axis = 1).to_numpy(),
            'movement_dur': data['movement_dur']
        }

    return(iso_dict)



#### Signal

In [ ]:
session1_data = get_task_data(session1_folder, 8, movement_names, 'move')
session1_full = get_move(session1_data, samp_freq)
session1_iso4 = get_move(session1_data, samp_freq, win = 4)

session1_fist_iso4 = session1_iso4['fist']['combined_emg_reps']
session1_fist_full = session1_full['fist']['combined_emg_reps']

session1_ft_iso4 = session1_iso4['fast_tripod']['combined_emg_reps']
session1_ft_full = session1_full['fast_tripod']['combined_emg_reps']

session1_ffe_iso4 = session1_iso4['fast_finger_ext']['combined_emg_reps']
session1_ffe_full = session1_full['fast_finger_ext']['combined_emg_reps']

#### Noise

In [ ]:
session1_rest_load = get_task_data(session1_folder, 8, movement_names, 'rest')
session1_rest = get_move(session1_rest_load, samp_freq)

session1_fist_rest = session1_rest['fist']['combined_emg_reps']
session1_ft_rest = session1_rest['fast_tripod']['combined_emg_reps']
session1_ffe_rest = session1_rest['fast_finger_ext']['combined_emg_reps']

In [80]:
print(session1_fist_rest[0:5])

[[ -7.408236  -13.659629  -18.829685  ...   5.1129193  11.703414
   12.325697 ]
 [ -8.478871  -16.083435  -22.890522  ...   9.067921   16.744312
   18.866764 ]
 [-11.018106  -20.623018  -28.72919   ...  11.141033   18.501291
   20.42593  ]
 [-14.156023  -25.30214   -32.750656  ...   9.222504   13.578265
   13.644588 ]
 [-13.912285  -23.46608   -29.465664  ...   4.7282205   7.5474253
    7.086979 ]]


In [38]:
from scipy.signal import welch

def calc_PSD(sig, fsamp=2048, nperseg=2048, noverlap=2048/2):
    '''
    Compute the power spectral density for each channel using Welch's method.

    Args:
        sig (ndarray): Multi-channel signal (Channels x Samples)
        fsamp (float): Sampling rate in Hz
        nperseg (int): Number of data points per segment
        overlap (int): Number of overlapping samples
    '''

    f, _ = welch(sig[0,:], fs=fsamp, nperseg=nperseg, noverlap=noverlap)

    P = np.zeros((sig.shape[0], f.shape[0]))

    for i in range(sig.shape[0]):
        _ , P[i,:] = welch(sig[i,:], fs=fsamp, nperseg=nperseg, noverlap=noverlap)

    return P, f


In [ ]:
# Calculate the psd for signal and noise
P_s, f_s = calc_PSD(session1_fist_full, fsamp=samp_freq, nperseg=samp_freq, noverlap=samp_freq/2)
P_r, f_r = calc_PSD(session1_fist_rest, fsamp=samp_freq, nperseg=samp_freq, noverlap=samp_freq/2)

# Calculate the SNR
p_sig = np.mean(session1_fist_full**2)
p_noise = np.mean(session1_fist_rest**2)
snr = 10 * np.log10(p_sig / p_noise)
val = np.array2string(snr, formatter={'float_kind': lambda x: f"{x:.2f}"})
print(f"The SNR of the raw signal is {val} dB.") # 0909 snr was 1.06 for fist -> extremely low

The SNR of the raw signal is 1.06 dB.


In [ ]:
#Important: if you are in a jupyter notebook,
#this line allows the plots to be displayed in a separate window instead of inline
#which is needed for the mask and channel reviews gui
%matplotlib qt

from src.utils.select_windows import select_windows
from src.utils.review_channels import review_channels

In [10]:
good_mask_path = "demo_good_mask.npy"
mask_path = "demo_mask.npy"
exclude_path = "demo_exclude.npy"

In [ ]:
review_channels(session1_fist_emg, good_mask_path, fs=samp_freq, label="demo")

seed: auto RMS-outlier mask (32/32 good, bad [])


In [ ]:
review_channels(session1_fist_clean, good_mask_path, fs=samp_freq, label="demo")

In [69]:
from scipy.interpolate import interp1d

def remove_spikes(emg, threshold_std=5):
    """
    Remove large spikes via simple thresholding and linear interpolation.
    
    Parameters
    ----------
    emg : ndarray, shape (n_channels, n_samples)
    threshold_std : float
        Number of standard deviations for threshold
    
    Returns
    -------
    emg_clean : ndarray
    spike_mask : ndarray, bool
    """
    emg_clean = emg.copy()
    spike_mask = np.zeros_like(emg, dtype=bool)
    
    for ch in range(emg.shape[0]):
        signal = emg[ch]
        threshold = threshold_std * np.std(signal)
        
        spikes = np.abs(signal) > threshold
        spike_mask[ch] = spikes
        
        if np.any(spikes):
            clean_idx = np.where(~spikes)[0]
            spike_idx = np.where(spikes)[0]
            
            if len(clean_idx) > 1:
                interp = interp1d(clean_idx, signal[clean_idx], 
                                  kind='linear', bounds_error=False, 
                                  fill_value='extrapolate')
                emg_clean[ch, spike_idx] = interp(spike_idx)
    
    return emg_clean, spike_mask


In [70]:
session1_fist_full_clean, spikes = remove_spikes(session1_fist_full, threshold_std=3)


seed: auto RMS-outlier mask (32/32 good, bad [])


In [71]:
# Custom modules
from src.muniverse.algorithms.decomposition import decompose_cbss

# Load the baseline configuration
with open(os.path.join(REPO_DIR, 'src', 'configs', 'cbss.json')) as f:
    cbss_config = json.load(f)['Config']

# The JSON stores disabled options as the string "None", turn them back into real None
cbss_config = {k: (None if v == "None" else v) for k, v in cbss_config.items()}

# # Fewer iterations than the config asks for, again just to keep the notebook quick
# cbss_config['ica_n_iter'] = 30

# no MUs originally found, lower exp:
cbss_config['opt_function_exp'] = 2

print(json.dumps(cbss_config, indent=2))

{
  "start_time": 0,
  "end_time": -1,
  "sampling_frequency": 2048,
  "bandpass": null,
  "bandpass_order": 2,
  "notch_frequency": 50,
  "notch_n_harmonics": 3,
  "notch_order": 2,
  "notch_width": 1,
  "ext_fact": 16,
  "whitening_method": "ZCA",
  "whitening_reg": "auto",
  "ica_n_iter": 100,
  "opt_initalization": "random",
  "opt_function_exp": 2,
  "opt_max_iter": 100,
  "opt_tol": 0.0001,
  "source_deflation": "gram-schmidt",
  "peel_off": true,
  "fr_peeloff": true,
  "cluster_method": "kmeans",
  "random_seed": 1909,
  "refinement_loop": true,
  "sil_th": 0.85,
  "cov_th": 0.35,
  "verbose_mode": true
}


In [72]:

#Decomposition/Processing Imports
from src.muniverse.algorithms.cbss import CBSS
# Everything from step 1 to step 8, in one call
decomposer = CBSS(**cbss_config)

sources_cbss, spikes_cbss, sil_cbss, filters_cbss, Z_cbss, centroids_cbss = decomposer.decompose(
    session1_fist_full_clean, fsamp=fs
    
)

print(f"\nCBSS found {sources_cbss.shape[0]} motor units, ")

[INFO] Extending signals by factor 16...

STARTING CBSS DECOMPOSITION
Max iterations: 100, Silhouette threshold: 0.850, CoV threshold: 0.350

FR peel-off enabled
FR peel-off enabled
Refinement loop for source 2 (initial Sil: 0.929, CoV: 2.478, Spikes: 303)
FR peel-off enabled
Refinement loop for source 3 (initial Sil: 0.915, CoV: 2.559, Spikes: 316)
FR peel-off enabled
Refinement loop for source 4 (initial Sil: 0.911, CoV: 2.461, Spikes: 315)
FR peel-off enabled
Refinement loop for source 5 (initial Sil: 0.872, CoV: 2.110, Spikes: 233)
FR peel-off enabled
Refinement loop for source 6 (initial Sil: 0.912, CoV: 1.628, Spikes: 15)
FR peel-off enabled
Refinement loop for source 7 (initial Sil: 0.931, CoV: 2.507, Spikes: 309)
FR peel-off enabled
Refinement loop for source 8 (initial Sil: 0.875, CoV: 2.241, Spikes: 345)
FR peel-off enabled
Refinement loop for source 9 (initial Sil: 0.939, CoV: 2.483, Spikes: 304)
FR peel-off enabled
Refinement loop for source 10 (initial Sil: 0.878, CoV: 2.1